In [ ]:
# ANOTASI GAMBAR pada Model SSD MobileNet & YOLO

In [1]:
# Periksa keberadaan folder dan file dataset
from pathlib import Path
import tensorflow as tf

# Lokasi dataset YOLO
YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")
YOLO_IMAGES = YOLO_ROOT / "images/test"
YOLO_LABELS = YOLO_ROOT / "labels/test"

# Lokasi TFRecord SSD
TEST_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record"
)

# 1. Pastikan semua lokasi tersedia
paths = {
    "Folder gambar YOLO": YOLO_IMAGES,
    "Folder label YOLO": YOLO_LABELS,
    "TFRecord test SSD": TEST_RECORD,
}

print("=== A. PEMERIKSAAN LOKASI ===")
for name, path in paths.items():
    print(f"{name}: {'ADA' if path.exists() else 'TIDAK ADA'}")
    print(path)

assert YOLO_IMAGES.is_dir(), "Folder gambar YOLO tidak ditemukan."
assert YOLO_LABELS.is_dir(), "Folder label YOLO tidak ditemukan."
assert TEST_RECORD.is_file(), "test.record tidak ditemukan."

# 2. Hitung gambar dan label YOLO
image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

yolo_images = sorted(
    p.name for p in YOLO_IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
)
yolo_labels = sorted(
    p.name for p in YOLO_LABELS.iterdir()
    if p.is_file() and p.suffix.lower() == ".txt"
)

print("\n=== B. JUMLAH FILE YOLO ===")
print("Jumlah gambar:", len(yolo_images))
print("Jumlah label TXT:", len(yolo_labels))

# 3. Baca nama gambar yang tersimpan di TFRecord
feature_description = {
    "image/filename": tf.io.FixedLenFeature(
        [], tf.string, default_value=b""
    ),
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
}

tfrecord_filenames = []

for raw in tf.data.TFRecordDataset(str(TEST_RECORD)):
    example = tf.io.parse_single_example(raw, feature_description)
    filename = example["image/filename"].numpy().decode(
        "utf-8", errors="replace"
    )
    tfrecord_filenames.append(Path(filename).name)

print("\n=== C. JUMLAH DATA TEST SSD ===")
print("Jumlah record:", len(tfrecord_filenames))
print("Nama file kosong:", sum(not x for x in tfrecord_filenames))
print("Nama file duplikat:", len(tfrecord_filenames) -
      len(set(tfrecord_filenames)))

# 4. Bandingkan nama gambar YOLO dengan TFRecord
yolo_image_set = set(yolo_images)
tfrecord_image_set = set(tfrecord_filenames)

print("\n=== D. PERBANDINGAN NAMA GAMBAR ===")
print("Gambar YOLO tidak ada di TFRecord:",
      len(yolo_image_set - tfrecord_image_set))
print("Gambar TFRecord tidak ada di YOLO:",
      len(tfrecord_image_set - yolo_image_set))

if yolo_image_set - tfrecord_image_set:
    print("\nContoh gambar YOLO yang tidak ditemukan di TFRecord:")
    print(sorted(yolo_image_set - tfrecord_image_set)[:10])

if tfrecord_image_set - yolo_image_set:
    print("\nContoh gambar TFRecord yang tidak ditemukan di YOLO:")
    print(sorted(tfrecord_image_set - yolo_image_set)[:10])

print("\n=== E. STATUS PEMERIKSAAN AWAL ===")
if (
    len(yolo_images) == 340
    and len(yolo_labels) == 340
    and len(tfrecord_filenames) == 340
    and len(set(tfrecord_filenames)) == 340
    and yolo_image_set == tfrecord_image_set
):
    print("LOLOS: jumlah dan nama gambar cocok.")
    print("Catatan: kesamaan isi gambar dan anotasi belum diverifikasi.")
else:
    print("PERLU DIPERIKSA: ada jumlah atau nama file yang berbeda.")

2026-10-26 10:16:13.912504: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-26 10:16:15.530320: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/rzangef/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:223: FutureWarning: You are using a non-supported Python version (3.10.12). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
2026-10-26 10:16:22.458676: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will n

=== A. PEMERIKSAAN LOKASI ===
Folder gambar YOLO: ADA
/home/rzangef/spark-data/CHVG-YOLO/images/test
Folder label YOLO: ADA
/home/rzangef/spark-data/CHVG-YOLO/labels/test
TFRecord test SSD: ADA
/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record

=== B. JUMLAH FILE YOLO ===
Jumlah gambar: 340
Jumlah label TXT: 340


2026-10-26 10:16:27.044023: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-10-26 10:16:27.146148: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144



=== C. JUMLAH DATA TEST SSD ===
Jumlah record: 340
Nama file kosong: 0
Nama file duplikat: 0

=== D. PERBANDINGAN NAMA GAMBAR ===
Gambar YOLO tidak ada di TFRecord: 0
Gambar TFRecord tidak ada di YOLO: 0

=== E. STATUS PEMERIKSAAN AWAL ===
LOLOS: jumlah dan nama gambar cocok.
Catatan: kesamaan isi gambar dan anotasi belum diverifikasi.


2026-10-26 10:16:27.662385: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [2]:
# Membandingkan anotasi YOLO dengan anotasi SSD
from pathlib import Path
import tensorflow as tf
import numpy as np
from collections import Counter
import hashlib

YOLO_IMAGES = Path("/home/rzangef/spark-data/CHVG-YOLO/images/test")
YOLO_LABELS = Path("/home/rzangef/spark-data/CHVG-YOLO/labels/test")
TEST_RECORD = Path(
    "/home/rzangef/SSD-PPE/dataset/tfrecord_yolo_split/test.record"
)

FEATURES = {
    "image/filename": tf.io.FixedLenFeature([], tf.string, default_value=b""),
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

def dense(example, key, dtype):
    return tf.sparse.to_dense(example[key]).numpy().astype(dtype)

def file_sha256(data):
    return hashlib.sha256(data).hexdigest()

# Baca semua label dari YOLO
yolo_annotations = {}

for image_name in sorted(p.name for p in YOLO_IMAGES.iterdir()
                         if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}):
    label_path = YOLO_LABELS / (Path(image_name).stem + ".txt")
    objects = []

    with open(label_path, "r") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            if len(parts) != 5:
                raise ValueError(
                    f"Format label tidak valid: {label_path}, baris {line_number}"
                )

            cls, xc, yc, w, h = map(float, parts)
            # Konversi YOLO [class, xc, yc, w, h]
            # menjadi [class, xmin, ymin, xmax, ymax]
            objects.append([
                int(cls),
                xc - w / 2,
                yc - h / 2,
                xc + w / 2,
                yc + h / 2,
            ])

    yolo_annotations[image_name] = np.array(
        objects, dtype=np.float64
    ).reshape(-1, 5)

# Baca anotasi dan gambar yang tersimpan di TFRecord
ssd_annotations = {}
image_hash_matches = {}
record_count = 0

for raw in tf.data.TFRecordDataset(str(TEST_RECORD)):
    example = tf.io.parse_single_example(raw, FEATURES)
    name = Path(
        example["image/filename"].numpy().decode("utf-8")
    ).name

    if name in ssd_annotations:
        raise ValueError(f"Record duplikat: {name}")

    labels = dense(example, "image/object/class/label", np.int64)
    xmin = dense(example, "image/object/bbox/xmin", np.float64)
    xmax = dense(example, "image/object/bbox/xmax", np.float64)
    ymin = dense(example, "image/object/bbox/ymin", np.float64)
    ymax = dense(example, "image/object/bbox/ymax", np.float64)

    boxes = np.column_stack([labels, xmin, ymin, xmax, ymax])
    ssd_annotations[name] = boxes

    # Periksa apakah byte gambar TFRecord identik dengan file YOLO
    image_path = YOLO_IMAGES / name
    encoded = example["image/encoded"].numpy()

    if image_path.exists():
        image_hash_matches[name] = (
            file_sha256(encoded) == file_sha256(image_path.read_bytes())
        )
    else:
        image_hash_matches[name] = False

    record_count += 1

# Bandingkan anotasi tanpa mengandalkan urutan objek dalam file
TOLERANCE = 1e-3
matched_images = 0
images_with_differences = []
total_yolo_objects = 0
total_ssd_objects = 0
class_count_yolo = Counter()
class_count_ssd = Counter()

for name, yolo_boxes in yolo_annotations.items():
    if name not in ssd_annotations:
        images_with_differences.append((name, "Tidak ada di TFRecord"))
        continue

    ssd_boxes = ssd_annotations[name]
    total_yolo_objects += len(yolo_boxes)
    total_ssd_objects += len(ssd_boxes)

    for row in yolo_boxes:
        class_count_yolo[int(row[0])] += 1

    for row in ssd_boxes:
        class_count_ssd[int(row[0])] += 1

    # YOLO memakai ID kelas 0–7; label SSD/TFRecord memakai 1–8.
    # Cocokkan kelas dan koordinat, terlepas dari urutan anotasinya.
    used_ssd = set()
    mismatch = []

    if len(yolo_boxes) != len(ssd_boxes):
        mismatch.append(
            f"jumlah objek berbeda (YOLO {len(yolo_boxes)}, SSD {len(ssd_boxes)})"
        )

    for ybox in yolo_boxes:
        y_class = int(ybox[0])
        y_coords = ybox[1:]
        candidates = []

        for j, sbox in enumerate(ssd_boxes):
            if j in used_ssd:
                continue
            if int(sbox[0]) != y_class + 1:
                continue

            max_diff = np.max(np.abs(y_coords - sbox[1:]))
            if max_diff <= TOLERANCE:
                candidates.append((max_diff, j))

        if candidates:
            _, best_j = min(candidates)
            used_ssd.add(best_j)
        else:
            mismatch.append(
                f"objek kelas YOLO {y_class} tidak punya pasangan koordinat"
            )

    if len(used_ssd) != len(ssd_boxes):
        mismatch.append("ada objek SSD yang belum cocok")

    if mismatch:
        images_with_differences.append((name, "; ".join(mismatch)))
    else:
        matched_images += 1

# Ringkasan
print("=== PEMERIKSAAN ISI GAMBAR ===")
print("Gambar dengan byte identik:",
      sum(image_hash_matches.values()), "/", len(image_hash_matches))
print("Gambar dengan byte berbeda/tidak ditemukan:",
      len(image_hash_matches) - sum(image_hash_matches.values()))

print("\n=== PEMERIKSAAN ANOTASI ===")
print("Record SSD:", record_count)
print("Gambar YOLO:", len(yolo_annotations))
print("Gambar dengan anotasi yang cocok:",
      matched_images, "/", len(yolo_annotations))
print("Gambar dengan perbedaan anotasi:", len(images_with_differences))
print("Total objek YOLO:", total_yolo_objects)
print("Total objek SSD:", total_ssd_objects)
print("Jumlah objek per kelas YOLO (ID 0–7):", dict(sorted(class_count_yolo.items())))
print("Jumlah objek per kelas SSD (ID 1–8):", dict(sorted(class_count_ssd.items())))

if images_with_differences:
    print("\n=== CONTOH PERBEDAAN (maksimal 10) ===")
    for name, reason in images_with_differences[:10]:
        print(name, "->", reason)

print("\n=== KESIMPULAN ===")
if (
    len(image_hash_matches) == 340
    and all(image_hash_matches.values())
    and len(yolo_annotations) == 340
    and record_count == 340
    and matched_images == 340
    and total_yolo_objects == total_ssd_objects
):
    print("LOLOS: byte gambar dan anotasi cocok untuk seluruh data test.")
else:
    print("BELUM LOLOS SEPENUHNYA: periksa ringkasan dan contoh perbedaan di atas.")

=== PEMERIKSAAN ISI GAMBAR ===
Gambar dengan byte identik: 340 / 340
Gambar dengan byte berbeda/tidak ditemukan: 0

=== PEMERIKSAAN ANOTASI ===
Record SSD: 340
Gambar YOLO: 340
Gambar dengan anotasi yang cocok: 338 / 340
Gambar dengan perbedaan anotasi: 2
Total objek YOLO: 2385
Total objek SSD: 2385
Jumlah objek per kelas YOLO (ID 0–7): {0: 964, 1: 430, 2: 247, 3: 263, 4: 103, 5: 145, 6: 82, 7: 151}
Jumlah objek per kelas SSD (ID 1–8): {1: 964, 2: 430, 3: 247, 4: 263, 5: 103, 6: 145, 7: 82, 8: 151}

=== CONTOH PERBEDAAN (maksimal 10) ===
ppe_0033_jpg.rf.0f3120466c46e5ed667e39a96c7c1821.jpg -> objek kelas YOLO 0 tidak punya pasangan koordinat; ada objek SSD yang belum cocok
ppe_0948_jpg.rf.44b1f52f34855bbd84e9a8a01c4a1725.jpg -> objek kelas YOLO 0 tidak punya pasangan koordinat; objek kelas YOLO 5 tidak punya pasangan koordinat; ada objek SSD yang belum cocok

=== KESIMPULAN ===
BELUM LOLOS SEPENUHNYA: periksa ringkasan dan contoh perbedaan di atas.


2026-10-26 10:18:37.304849: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [3]:
# Dari pemeriksaan diatas ditemukan perbedaan pada 2 anotasi gambar
# Cek selisih koordinat
import numpy as np
from pathlib import Path

TARGET_IMAGES = [
    "ppe_0033_jpg.rf.0f3120466c46e5ed667e39a96c7c1821.jpg",
    "ppe_0948_jpg.rf.44b1f52f34855bbd84e9a8a01c4a1725.jpg",
]

TOLERANCE = 1e-3

for name in TARGET_IMAGES:
    print("\n" + "=" * 80)
    print("GAMBAR:", name)

    yolo_boxes = yolo_annotations[name]
    ssd_boxes = ssd_annotations[name]

    for y in yolo_boxes:
        cls_yolo = int(y[0])
        coords_yolo = y[1:]

        candidates = [
            (j, s)
            for j, s in enumerate(ssd_boxes)
            if int(s[0]) == cls_yolo + 1
        ]

        print(f"\nKelas YOLO {cls_yolo} (kelas SSD {cls_yolo + 1})")
        print("Koordinat YOLO [xmin, ymin, xmax, ymax]:")
        print(np.round(coords_yolo, 7))

        if not candidates:
            print("Tidak ada objek SSD dengan kelas yang sama.")
            continue

        differences = [
            (float(np.max(np.abs(coords_yolo - s[1:]))), j, s)
            for j, s in candidates
        ]
        diff, best_j, best_ssd = min(differences, key=lambda x: x[0])

        print("Koordinat SSD terdekat:")
        print(np.round(best_ssd[1:], 7))
        print(f"Selisih koordinat maksimum: {diff:.8f}")
        print("Cocok dalam toleransi:", diff <= TOLERANCE)


GAMBAR: ppe_0033_jpg.rf.0f3120466c46e5ed667e39a96c7c1821.jpg

Kelas YOLO 3 (kelas SSD 4)
Koordinat YOLO [xmin, ymin, xmax, ymax]:
[0.21875  0.095312 0.373438 0.30625 ]
Koordinat SSD terdekat:
[0.21875   0.0953125 0.3734375 0.30625  ]
Selisih koordinat maksimum: 0.00000051
Cocok dalam toleransi: True

Kelas YOLO 3 (kelas SSD 4)
Koordinat YOLO [xmin, ymin, xmax, ymax]:
[0.9343745 0.1343745 0.9999995 0.2937495]
Koordinat SSD terdekat:
[0.934375 0.134375 1.       0.29375 ]
Selisih koordinat maksimum: 0.00000051
Cocok dalam toleransi: True

Kelas YOLO 4 (kelas SSD 5)
Koordinat YOLO [xmin, ymin, xmax, ymax]:
[0.58125   0.1890625 0.7       0.2656255]
Koordinat SSD terdekat:
[0.58125   0.1890625 0.7       0.265625 ]
Selisih koordinat maksimum: 0.00000050
Cocok dalam toleransi: True

Kelas YOLO 3 (kelas SSD 4)
Koordinat YOLO [xmin, ymin, xmax, ymax]:
[0.8750005 0.151563  0.9718755 0.332813 ]
Koordinat SSD terdekat:
[0.875     0.1515625 0.971875  0.3328125]
Selisih koordinat maksimum: 0.0000005

In [4]:
# Uji clipping untuk mengatasi masalah perbedaan anotasi

import numpy as np

TOLERANCE = 1e-3
matched_images_clipped = 0
differences_clipped = []

for name, yolo_boxes in yolo_annotations.items():
    if name not in ssd_annotations:
        differences_clipped.append((name, "Tidak ada di TFRecord"))
        continue

    # Salin anotasi, lalu batasi koordinat ke rentang 0–1.
    yolo_clipped = yolo_boxes.copy()
    yolo_clipped[:, 1:5] = np.clip(yolo_clipped[:, 1:5], 0.0, 1.0)

    ssd_boxes = ssd_annotations[name]
    used_ssd = set()
    mismatch_count = 0

    for ybox in yolo_clipped:
        cls_yolo = int(ybox[0])
        coords_yolo = ybox[1:]

        candidates = []
        for j, sbox in enumerate(ssd_boxes):
            if j in used_ssd:
                continue
            if int(sbox[0]) != cls_yolo + 1:
                continue

            diff = np.max(np.abs(coords_yolo - sbox[1:]))
            if diff <= TOLERANCE:
                candidates.append((diff, j))

        if candidates:
            _, best_j = min(candidates)
            used_ssd.add(best_j)
        else:
            mismatch_count += 1

    if mismatch_count == 0 and len(used_ssd) == len(ssd_boxes):
        matched_images_clipped += 1
    else:
        differences_clipped.append(
            (name, f"Objek YOLO tidak berpasangan: {mismatch_count}; "
                   f"objek SSD belum cocok: {len(ssd_boxes) - len(used_ssd)}")
        )

print("=== HASIL SETELAH CLIPPING SEMENTARA ===")
print(f"Anotasi cocok: {matched_images_clipped}/{len(yolo_annotations)}")
print(f"Gambar yang masih berbeda: {len(differences_clipped)}")
for item in differences_clipped[:10]:
    print(item)

if matched_images_clipped == 340 and not differences_clipped:
    print("KESIMPULAN: SELURUH ANOTASI COCOK SETELAH CLIPPING.")
else:
    print("KESIMPULAN: MASIH ADA PERBEDAAN YANG PERLU DIPERIKSA.")

=== HASIL SETELAH CLIPPING SEMENTARA ===
Anotasi cocok: 340/340
Gambar yang masih berbeda: 0
KESIMPULAN: SELURUH ANOTASI COCOK SETELAH CLIPPING.


In [ ]:
# Memastikan evaluasi akhir SSD & YOLO memakai data uji serta ground truth yang konsisten

In [5]:
# Menetapkan ground truth. Acuan label YOLO dengan ret]ntang koordinat kotak 0-1

import numpy as np
from collections import Counter

# Ground truth acuan: anotasi YOLO dengan koordinat yang dibatasi 0–1.
canonical_gt = {}

for name, boxes in yolo_annotations.items():
    boxes = boxes.copy()
    boxes[:, 1:5] = np.clip(boxes[:, 1:5], 0.0, 1.0)
    canonical_gt[name] = boxes

# Ringkasan ground truth acuan
class_counts = Counter()
total_objects = 0
invalid_boxes = []

for name, boxes in canonical_gt.items():
    total_objects += len(boxes)

    for box in boxes:
        cls = int(box[0])
        class_counts[cls] += 1

        coords = box[1:5]
        xmin, ymin, xmax, ymax = coords

        if not (
            np.all(np.isfinite(coords))
            and 0 <= xmin < xmax <= 1
            and 0 <= ymin < ymax <= 1
        ):
            invalid_boxes.append((name, box.tolist()))

# Verifikasi ground truth acuan terhadap anotasi TFRecord SSD
matched_images = 0
mismatches = []

for name, gt_boxes in canonical_gt.items():
    if name not in ssd_annotations:
        mismatches.append((name, "Gambar tidak ada di TFRecord"))
        continue

    ssd_boxes = ssd_annotations[name]
    used_ssd = set()
    all_matched = True

    for gt in gt_boxes:
        cls_yolo = int(gt[0])
        coords = gt[1:5]
        candidates = []

        for j, ssd_box in enumerate(ssd_boxes):
            if j in used_ssd:
                continue
            if int(ssd_box[0]) != cls_yolo + 1:
                continue

            diff = np.max(np.abs(coords - ssd_box[1:5]))
            if diff <= 1e-3:
                candidates.append((diff, j))

        if candidates:
            _, best_j = min(candidates)
            used_ssd.add(best_j)
        else:
            all_matched = False

    if all_matched and len(used_ssd) == len(ssd_boxes):
        matched_images += 1
    else:
        mismatches.append((name, "Jumlah atau koordinat objek berbeda"))

print("=== GROUND TRUTH ACUAN ===")
print("Jumlah gambar:", len(canonical_gt))
print("Jumlah objek:", total_objects)
print("Jumlah objek per kelas YOLO (ID 0–7):", dict(sorted(class_counts.items())))
print("Kotak tidak valid:", len(invalid_boxes))
print("Anotasi cocok dengan TFRecord:", f"{matched_images}/{len(canonical_gt)}")
print("Gambar yang masih berbeda:", len(mismatches))

if (
    len(canonical_gt) == 340
    and total_objects == 2385
    and len(class_counts) == 8
    and len(invalid_boxes) == 0
    and matched_images == 340
    and len(mismatches) == 0
):
    print("\nSTATUS: LULUS — ground truth acuan siap digunakan.")
else:
    print("\nSTATUS: BELUM LULUS — periksa hasil pemeriksaan di atas.")

=== GROUND TRUTH ACUAN ===
Jumlah gambar: 340
Jumlah objek: 2385
Jumlah objek per kelas YOLO (ID 0–7): {0: 964, 1: 430, 2: 247, 3: 263, 4: 103, 5: 145, 6: 82, 7: 151}
Kotak tidak valid: 0
Anotasi cocok dengan TFRecord: 340/340
Gambar yang masih berbeda: 0

STATUS: LULUS — ground truth acuan siap digunakan.


In [6]:
# Periksa checkpoint model Yolo (40 epoch), SSD (ckpt-7)
from pathlib import Path

# Kandidat checkpoint YOLO 40 epoch
YOLO_CANDIDATES = [
    Path("/home/rzangef/spark-data/yolo11_runs/yolo11s_40epochs/weights/best.pt"),
    Path("/home/rzangef/spark-data/yolo11_runs/yolo11s_40epochs/weights/last.pt"),
]

# Checkpoint SSD yang telah dilatih
SSD_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_yolo_split"
)
SSD_CHECKPOINT = SSD_DIR / "ckpt-7"
SSD_CHECKPOINT_INDEX = Path(str(SSD_CHECKPOINT) + ".index")

# Konfigurasi evaluasi SSD
SSD_TEST_CONFIG = Path(
    "/home/rzangef/SSD-PPE/models/"
    "pipeline_ppe_yolo_split_test.config"
)

print("=== CHECKPOINT YOLO ===")
for path in YOLO_CANDIDATES:
    print(f"{path}")
    print("  Tersedia:", path.is_file())

print("\n=== CHECKPOINT SSD ===")
print("Checkpoint:", SSD_CHECKPOINT)
print("File index tersedia:", SSD_CHECKPOINT_INDEX.is_file())
print("File data checkpoint tersedia:", any(
    SSD_DIR.glob("ckpt-7.data-*")
))

print("\n=== KONFIGURASI TEST SSD ===")
print(SSD_TEST_CONFIG)
print("Tersedia:", SSD_TEST_CONFIG.is_file())

=== CHECKPOINT YOLO ===
/home/rzangef/spark-data/yolo11_runs/yolo11s_40epochs/weights/best.pt
  Tersedia: True
/home/rzangef/spark-data/yolo11_runs/yolo11s_40epochs/weights/last.pt
  Tersedia: True

=== CHECKPOINT SSD ===
Checkpoint: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7
File index tersedia: True
File data checkpoint tersedia: True

=== KONFIGURASI TEST SSD ===
/home/rzangef/SSD-PPE/models/pipeline_ppe_yolo_split_test.config
Tersedia: True


In [ ]:
# Melakukan Pengujian YOLO

In [7]:
# Uji format prediksi pada satu gambar
from pathlib import Path
import sys
import numpy as np
import tensorflow as tf
from google.protobuf import text_format
from ultralytics import YOLO

# 1. Satu gambar yang ada di dataset test
TEST_IMAGES = Path("/home/rzangef/spark-data/CHVG-YOLO/images/test")
image_name = sorted(canonical_gt.keys())[0]
image_path = TEST_IMAGES / image_name

assert image_path.is_file(), f"Gambar tidak ditemukan: {image_path}"

# 2. Muat YOLOv11s 40 epoch
YOLO_WEIGHTS = Path(
    "/home/rzangef/spark-data/yolo11_runs/"
    "yolo11s_40epochs/weights/best.pt"
)
yolo_model = YOLO(str(YOLO_WEIGHTS))

# Confidence rendah agar kita melihat format prediksi, bukan hasil akhir metrik
yolo_result = yolo_model.predict(
    source=str(image_path),
    imgsz=640,
    conf=0.001,
    verbose=False
)[0]

print("=== YOLO ===")
print("Nama kelas:", yolo_result.names)
print("Jumlah prediksi:", len(yolo_result.boxes))

if len(yolo_result.boxes):
    print("Contoh class ID:", yolo_result.boxes.cls[:10].cpu().numpy().astype(int))
    print("Contoh confidence:", np.round(
        yolo_result.boxes.conf[:10].cpu().numpy(), 4
    ))
    print("Contoh koordinat xyxy piksel:", np.round(
        yolo_result.boxes.xyxy[:5].cpu().numpy(), 2
    ))

# 3. Muat model SSD dari konfigurasi test dan checkpoint ckpt-7
PROJECT = Path("/home/rzangef/SSD-PPE")
RESEARCH_DIR = str(PROJECT / "object-detection-api/research")
SSD_CONFIG = PROJECT / "models/pipeline_ppe_yolo_split_test.config"
SSD_CHECKPOINT = PROJECT / (
    "training/ssd_mobilenet_v2_fpnlite_320x320_yolo_split/ckpt-7"
)

if RESEARCH_DIR not in sys.path:
    sys.path.insert(0, RESEARCH_DIR)

from object_detection.protos import pipeline_pb2
from object_detection.builders import model_builder

pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()
with open(SSD_CONFIG, "r") as f:
    text_format.Merge(f.read(), pipeline_config)

detection_model = model_builder.build(
    model_config=pipeline_config.model,
    is_training=False
)

restore_status = tf.train.Checkpoint(
    model=detection_model
).restore(str(SSD_CHECKPOINT))
restore_status.expect_partial()

# 4. Inferensi SSD pada gambar yang sama
image_bytes = tf.io.read_file(str(image_path))
image = tf.io.decode_jpeg(image_bytes, channels=3)
input_tensor = tf.expand_dims(tf.cast(image, tf.float32), axis=0)

preprocessed_images, true_image_shapes = detection_model.preprocess(input_tensor)
prediction_dict = detection_model.predict(
    preprocessed_images, true_image_shapes
)
ssd_result = detection_model.postprocess(
    prediction_dict, true_image_shapes
)

ssd_scores = ssd_result["detection_scores"][0].numpy()
ssd_classes = ssd_result["detection_classes"][0].numpy().astype(int)
ssd_boxes = ssd_result["detection_boxes"][0].numpy()

# Hanya tampilkan 10 prediksi SSD dengan confidence tertinggi
order = np.argsort(ssd_scores)[::-1][:10]

print("\n=== SSD ===")
print("Jumlah prediksi yang dikembalikan:", len(ssd_scores))
print("Rentang class ID mentah:", (
    int(ssd_classes.min()), int(ssd_classes.max())
) if len(ssd_classes) else "kosong")
print("Confidence tertinggi:", round(float(ssd_scores.max()), 4))

print("\n10 prediksi SSD teratas:")
for i in order:
    print(
        f"class_id={ssd_classes[i]}, "
        f"confidence={ssd_scores[i]:.4f}, "
        f"box_yxyx_normalized={np.round(ssd_boxes[i], 4)}"
    )

print("\n=== GROUND TRUTH GAMBAR INI ===")
print("Jumlah objek:", len(canonical_gt[image_name]))
print("Class ID YOLO:", canonical_gt[image_name][:, 0].astype(int))

=== YOLO ===
Nama kelas: {0: 'person', 1: 'vest', 2: 'yellow', 3: 'white', 4: 'glass', 5: 'blue', 6: 'red', 7: 'head'}
Jumlah prediksi: 54
Contoh class ID: [7 7 7 0 7 7 0 0 0 0]
Contoh confidence: [     0.8552      0.8237      0.8025      0.8019      0.7892      0.7753      0.7459      0.7425      0.7323      0.7032]
Contoh koordinat xyxy piksel: [[     229.99      156.25      277.63      249.96]
 [     484.04      153.15      555.36      287.42]
 [     404.38      165.11       456.1      253.44]
 [     382.45      157.75      494.07      638.49]
 [     287.52      160.06      345.08      255.07]]


AttributeError: module 'keras._tf_keras.keras.layers' has no attribute 'experimental'

In [1]:
# Cek folder pengujian pada epoch 40
# Tujuan : untuk membedakan hasil training, evaluasi validasi, dan checkpoint model
from pathlib import Path

RUN_DIR = Path(
    "/home/rzangef/spark-data/yolo11_runs/yolo11s_40epochs"
)

print("Folder tersedia:", RUN_DIR.exists())

if RUN_DIR.exists():
    print("\nIsi folder hasil eksperimen:")
    for item in sorted(RUN_DIR.iterdir()):
        print(item.name, " [folder]" if item.is_dir() else " [file]")

    weights_dir = RUN_DIR / "weights"
    if weights_dir.exists():
        print("\nCheckpoint model:")
        for item in sorted(weights_dir.iterdir()):
            print(item.name, f"({item.stat().st_size / (1024**2):.2f} MB)")

Folder tersedia: True

Isi folder hasil eksperimen:
BoxF1_curve.png  [file]
BoxPR_curve.png  [file]
BoxP_curve.png  [file]
BoxR_curve.png  [file]
args.yaml  [file]
confusion_matrix.png  [file]
confusion_matrix_normalized.png  [file]
labels.jpg  [file]
results.csv  [file]
results.png  [file]
train_batch0.jpg  [file]
train_batch1.jpg  [file]
train_batch2.jpg  [file]
train_batch35670.jpg  [file]
train_batch35671.jpg  [file]
train_batch35672.jpg  [file]
val_batch0_labels.jpg  [file]
val_batch0_pred.jpg  [file]
val_batch1_labels.jpg  [file]
val_batch1_pred.jpg  [file]
val_batch2_labels.jpg  [file]
val_batch2_pred.jpg  [file]
weights  [folder]

Checkpoint model:
best.pt (18.29 MB)
last.pt (18.29 MB)


In [2]:
# Cek konfigurasi YOLO untuk mengetahui checkpoint atau dataset
from pathlib import Path
import yaml

RUN_DIR = Path(
    "/home/rzangef/spark-data/yolo11_runs/yolo11s_40epochs"
)

args = yaml.safe_load((RUN_DIR / "args.yaml").read_text())

for key in [
    "model", "data", "epochs", "imgsz",
    "batch", "device", "split", "conf", "iou", "max_det"
]:
    print(f"{key}: {args.get(key, '(tidak tercatat)')}")

print("\nCheckpoint best tersedia:",
      (RUN_DIR / "weights/best.pt").exists())

model: yolo11s.pt
data: /home/rzangef/spark-data/CHVG-YOLO/data.yaml
epochs: 40
imgsz: 640
batch: 1
device: cpu
split: val
conf: None
iou: 0.7
max_det: 300

Checkpoint best tersedia: True


In [3]:
# Evaluasi Test YOLO
from pathlib import Path
from ultralytics import YOLO
import numpy as np
import time

# Lokasi model dan dataset
RUN_DIR = Path(
    "/home/rzangef/spark-data/yolo11_runs/yolo11s_40epochs"
)
YOLO_ROOT = Path("/home/rzangef/spark-data/CHVG-YOLO")

MODEL_PATH = RUN_DIR / "weights/best.pt"
TEST_IMAGES = YOLO_ROOT / "images/test"
TEST_LABELS = YOLO_ROOT / "labels/test"

CONFIDENCE_THRESHOLD = 0.40
IOU_MATCH_THRESHOLD = 0.50

assert MODEL_PATH.exists(), f"Checkpoint tidak ditemukan: {MODEL_PATH}"
assert TEST_IMAGES.exists(), f"Folder gambar tidak ditemukan: {TEST_IMAGES}"
assert TEST_LABELS.exists(), f"Folder label tidak ditemukan: {TEST_LABELS}"

image_paths = sorted(TEST_IMAGES.glob("*.jpg"))
assert len(image_paths) == 340, (
    f"Jumlah gambar tidak sesuai: {len(image_paths)}"
)

print("Checkpoint:", MODEL_PATH)
print("Jumlah gambar test:", len(image_paths))
print("Confidence:", CONFIDENCE_THRESHOLD)
print("IoU pencocokan:", IOU_MATCH_THRESHOLD)

model = YOLO(str(MODEL_PATH))
print("Model YOLO berhasil dimuat.")

Checkpoint: /home/rzangef/spark-data/yolo11_runs/yolo11s_40epochs/weights/best.pt
Jumlah gambar test: 340
Confidence: 0.4
IoU pencocokan: 0.5
Model YOLO berhasil dimuat.


In [4]:
# Evaluasi YOLO menghasilkan precision, recall, f1-score
from pathlib import Path
import numpy as np
import time

# Konfigurasi evaluasi
CONF = 0.40
IOU_MATCH = 0.50

# Samakan batas deteksi dengan evaluasi SSD
NMS_IOU = 0.60
MAX_DET = 100

def read_yolo_labels(label_path):
    """Membaca label YOLO: class x_center y_center width height."""
    boxes = []
    classes = []

    if not label_path.exists():
        raise FileNotFoundError(f"Label tidak ditemukan: {label_path}")

    for line in label_path.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue

        cls, xc, yc, w, h = map(float, parts[:5])

        # Format kotak: [y1, x1, y2, x2], koordinat ternormalisasi
        box = [
            yc - h / 2,
            xc - w / 2,
            yc + h / 2,
            xc + w / 2
        ]
        boxes.append(box)
        classes.append(int(cls))

    return np.array(boxes, dtype=float).reshape(-1, 4), \
           np.array(classes, dtype=int)


def calculate_iou(box1, box2):
    """IoU untuk kotak [y1, x1, y2, x2]."""
    y1 = max(box1[0], box2[0])
    x1 = max(box1[1], box2[1])
    y2 = min(box1[2], box2[2])
    x2 = min(box1[3], box2[3])

    intersection = max(0.0, y2 - y1) * max(0.0, x2 - x1)

    area1 = max(0.0, box1[2] - box1[0]) * \
            max(0.0, box1[3] - box1[1])
    area2 = max(0.0, box2[2] - box2[0]) * \
            max(0.0, box2[3] - box2[1])

    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0.0


TP = 0
FP = 0
FN = 0
total_gt = 0
total_images = 0
start_time = time.time()

for idx, image_path in enumerate(image_paths, start=1):
    label_path = TEST_LABELS / f"{image_path.stem}.txt"

    gt_boxes, gt_classes = read_yolo_labels(label_path)
    total_gt += len(gt_boxes)

    # Prediksi satu gambar pada satu waktu agar penggunaan memori ringan
    result = model.predict(
        source=str(image_path),
        conf=CONF,
        iou=NMS_IOU,
        max_det=MAX_DET,
        imgsz=640,
        device="cpu",
        verbose=False
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        pred_boxes = np.empty((0, 4), dtype=float)
        pred_classes = np.empty(0, dtype=int)
        pred_scores = np.empty(0, dtype=float)
    else:
        # Konversi kotak prediksi piksel menjadi koordinat ternormalisasi
        xyxy = result.boxes.xyxy.cpu().numpy()
        pred_scores = result.boxes.conf.cpu().numpy()
        pred_classes = result.boxes.cls.cpu().numpy().astype(int)

        height, width = result.orig_shape
        pred_boxes = np.column_stack([
            xyxy[:, 1] / height,
            xyxy[:, 0] / width,
            xyxy[:, 3] / height,
            xyxy[:, 2] / width
        ])

    # Urutkan prediksi dari confidence tertinggi
    order = np.argsort(-pred_scores)
    pred_boxes = pred_boxes[order]
    pred_classes = pred_classes[order]

    matched_gt = set()

    # Pencocokan satu prediksi dengan maksimal satu ground truth
    for pbox, pclass in zip(pred_boxes, pred_classes):
        best_iou = 0.0
        best_gt_idx = None

        for gt_idx, (gt_box, gt_class) in enumerate(
            zip(gt_boxes, gt_classes)
        ):
            if gt_idx in matched_gt or pclass != gt_class:
                continue

            iou = calculate_iou(pbox, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_gt_idx is not None and best_iou >= IOU_MATCH:
            TP += 1
            matched_gt.add(best_gt_idx)
        else:
            FP += 1

    FN += len(gt_boxes) - len(matched_gt)
    total_images += 1

    if idx % 50 == 0 or idx == len(image_paths):
        print(f"Progres: {idx}/{len(image_paths)} gambar selesai")

precision = TP / (TP + FP) if TP + FP else 0.0
recall = TP / (TP + FN) if TP + FN else 0.0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0.0
)

print("\n===== HASIL EVALUASI MANUAL YOLOv11s =====")
print(f"Jumlah gambar test : {total_images}")
print(f"Ground truth       : {total_gt}")
print(f"TP                 : {TP}")
print(f"FP                 : {FP}")
print(f"FN                 : {FN}")
print(f"Precision          : {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall             : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-score           : {f1:.4f} ({f1*100:.2f}%)")
print(f"Waktu evaluasi     : {(time.time()-start_time)/60:.2f} menit")

Progres: 50/340 gambar selesai
Progres: 100/340 gambar selesai
Progres: 150/340 gambar selesai
Progres: 200/340 gambar selesai
Progres: 250/340 gambar selesai
Progres: 300/340 gambar selesai
Progres: 340/340 gambar selesai

===== HASIL EVALUASI MANUAL YOLOv11s =====
Jumlah gambar test : 340
Ground truth       : 2385
TP                 : 1953
FP                 : 178
FN                 : 432
Precision          : 0.9165 (91.65%)
Recall             : 0.8189 (81.89%)
F1-score           : 0.8649 (86.49%)
Waktu evaluasi     : 2.23 menit
